In [3]:
%pip install prophet

Note: you may need to restart the kernel to use updated packages.


### 1. Segment dataset

In [4]:
import pandas as pd
# load & prep
reg_cols = [
    "season_sin", "season_cos",
    "temp_avg", "precip_avg",
    "ap_price_weight",
    "possible_working_days", "avg_weight", "crude_price_weight"
]

df = pd.read_csv("../data/final_features_3.csv")
df["ds"] = pd.to_datetime(df["date"], format="%Y%m") + pd.offsets.MonthEnd(0)
df = df.rename(columns={"demand": "y"})
df[reg_cols] = df[reg_cols].astype(float)

cutoff = df["ds"].max() - pd.DateOffset(months=32)
df_train = df[df["ds"] <= cutoff].copy()
df_valid = df[df["ds"] > cutoff].copy()

### 2. Prophet baseline

In [5]:
from prophet import Prophet


# fit
m = Prophet(yearly_seasonality=True)
for reg in reg_cols:
    m.add_regressor(reg)
m.fit(df_train)


# Prophet in-sample predictions
p_train = m.predict(df_train[["ds"] + reg_cols])
p_valid = m.predict(df_valid[["ds"]+ reg_cols])

df_train["prophet_hat"] = p_train["yhat"].values
df_valid["prophet_hat"] = p_valid["yhat"].values

# # build future frame and attach regressors
# future = m.make_future_dataframe(periods=12, freq="M")
# future = future.merge(df[["ds"] + reg_cols], on="ds", how="left")

# # supply regressor values for the forecast horizon (replace with your own logic)
# for col in reg_cols:
#     future[col] = future[col].fillna(df[col].iloc[-12:].mean())

# forecast = m.predict(future)
# forecast.to_csv("../data/forecast_output.csv", index=False)


/Users/chungsangyoon/Desktop/Playground/ml_project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.
20:10:34 - cmdstanpy - INFO - Chain [1] start processing
20:10:34 - cmdstanpy - INFO - Chain [1] done processing


In [6]:
%pip install xgboost
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


### 3. XGB

In [7]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
# 1. Build residuals for XGB

df_train["resid"] = df_train["y"] - df_train["prophet_hat"]
df_valid["resid"] = df_valid["y"] - df_valid["prophet_hat"]

feat_cols = reg_cols
X_tr, y_tr = df_train[feat_cols], df_train["resid"]
X_va, y_va = df_valid[feat_cols], df_valid["resid"]


xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
    eval_metric="rmse",
    early_stopping_rounds=50,
)

xgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_va, y_va)],
    verbose=False,
)







,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,50
,enable_categorical,False
,eval_metric,'rmse'


### 4. Validation

In [8]:

import numpy as np
# Validation performance (hybrid = prophet + resid_correction)
valid_resid_pred = xgb_model.predict(X_va)
valid_hybrid = df_valid["prophet_hat"] + valid_resid_pred

from math import sqrt

mse = mean_squared_error(df_valid["y"], valid_hybrid)
rmse = sqrt(mse)
mae = mean_absolute_error(df_valid["y"], valid_hybrid)

# already have valid_hybrid and df_valid["y"]

# element-wise percentage error
pct_error = np.abs((df_valid["y"] - valid_hybrid) / df_valid["y"]) * 100

# Mean Absolute Percentage Error (MAPE)
mape = np.mean(pct_error)

# Root Mean Square Percentage Error (RMSPE)
rmspe = np.sqrt(np.mean(((df_valid["y"] - valid_hybrid) / df_valid["y"]) ** 2)) * 100

print(f"[Hybrid] RMSE={rmse:.1f}  MAE={mae:.1f}  MAPE={mape:.2f}%  RMSPE={rmspe:.2f}%")


[Hybrid] RMSE=21532.7  MAE=18508.1  MAPE=17.72%  RMSPE=21.29%
